In [3]:
import requests
from bs4 import BeautifulSoup
import re
import ast
import json
import pandas as pd
from pandas.io.json import json_normalize
from datetime import date

# Setup

In [4]:
# Request URL
page = requests.get("https://fastandfresh.in/collections/all")

In [5]:
# Fetch webpage
soup = BeautifulSoup(page.content,"html.parser")

In [6]:
# find number of pages on main site
page_links = soup.findAll("div", {"class": "pagination text-center"})
page_nums = [int(x.string) for x in page_links[0].findAll("span", {"class": "page"})]

pages = ['https://fastandfresh.in/collections/all?page='+str(x) for x in range(1,max(page_nums)+1)]

# Scrape

In [7]:
%%time

data_list = []

# for each page on items available
for p in pages:
    print (p)
    req = requests.get(p)
    psoup = BeautifulSoup(req.content,"html.parser")
    anchors = psoup.findAll("a", {"class": "product__image-wrapper product__image-wrapper--loading"})
    
    # search through each item's page
    for anchor in anchors:
        link = anchor['href']
        print (link)
        cont = BeautifulSoup(requests.get("https://fastandfresh.in"+link).content,"html.parser")
        data = json.loads(cont.findAll('script', type='application/ld+json')[2].string)
        data.update(data['offers'][0])
        data_list.append(data)
        #print (data)
    
    print ('\n')

https://fastandfresh.in/collections/all?page=1
/collections/all/products/amla
/collections/all/products/anar-kandhari
/collections/all/products/apple-golden-price-per-kg
/collections/all/products/apple-green-imported-price-per-kg
/collections/all/products/apple-royal-gala-price-per-500-gms
/collections/all/products/apple-shimla-price-per-500gms
/collections/all/products/apple-washington
/collections/all/products/apricot-dry-fresh-price-per-200gms
/collections/all/products/arbi-price-per-250-gms
/collections/all/products/asparagus-thai
/collections/all/products/avacado-hass-imported-price-per-pc
/collections/all/products/avacados
/collections/all/products/baby-corn-price-per-200gms
/collections/all/products/banana
/collections/all/products/banana-flower-price-per-pcs-approx-400gms-to-500gms
/collections/all/products/banana-raw-price-per-500-gms
/collections/all/products/basil-leaves
/collections/all/products/basket-ffg3
/collections/all/products/basket-ffg2
/collections/all/products/bas

# Look at data setup and save

In [8]:
df = pd.DataFrame(data_list)

In [9]:
df.shape

(162, 16)

In [10]:
df.columns

Index(['@context', '@type', 'name', 'brand', 'sku', 'mpn', 'description',
       'url', 'image', 'itemCondition', 'offers', 'price', 'priceCurrency',
       'availability', 'priceValidUntil', 'gtin14'],
      dtype='object')

In [11]:
dt = str(date.today())
dt

'2021-01-20'

In [12]:
df['date'] = dt

In [13]:
df.to_csv('scraped_data/FastnFresh_'+dt+'.csv',index=0)